# Specialist Model: Common Resistor Value Detection

This notebook trains the specialist YOLO model to directly classify common resistor values.

The workflow includes:
- Data loading
- Train/validation split
- Model training
- Evaluation
- Prediction and visualization
- Model export and deployment

In [ ]:
# Step 1: Load Dataset

from google.colab import drive
drive.mount('/content/gdrive')

!cp "/content/gdrive/MyDrive/The_Specialist.zip" /content/
!unzip -q /content/The_Specialist.zip -d /content/custom_data

## Data Preparation

Split dataset into training and validation sets (90/10).


In [ ]:
# Snapshot adapted from external YOLO workflow for dataset splitting

!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

!python /content/train_val_split.py --datapath="/content/custom_data" --train_pct=0.9

In [ ]:
# Install YOLO framework

!pip install ultralytics

## Configuration

Create YOLO data configuration file.

In [ ]:
import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

    if not os.path.exists(path_to_classes_txt):
        print("classes.txt file not found")
        return

    with open(path_to_classes_txt, "r") as f:
        classes = [line.strip() for line in f.readlines() if line.strip()]

    data = {
        "path": "/content/custom_data",
        "train": "train/images",
        "val": "validation/images",
        "nc": len(classes),
        "names": classes
    }

    with open(path_to_data_yaml, "w") as f:
        yaml.dump(data, f, sort_keys=False)

    print("data.yaml created")

path_to_classes_txt = "/content/custom_data/classes.txt"
path_to_data_yaml = "/content/data.yaml"

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

!cat /content/data.yaml

## Model Training

Train YOLO specialist model.

In [ ]:
!yolo detect train \
data=/content/data.yaml \
model=yolo11s.pt \
epochs=80 \
imgsz=640 \
batch=16 \
lr0=0.005

## Model Evaluation

Evaluate performance using standard metrics (mAP, Precision, Recall).

In [ ]:
!yolo detect val \
model=/content/runs/detect/train/weights/best.pt \
data=/content/data.yaml

## Prediction

Run inference on validation images.

In [ ]:
!yolo detect predict \
model=/content/runs/detect/train/weights/best.pt \
source=/content/custom_data/validation/images \
save=True

## Visualization

Display prediction results for qualitative inspection.

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

image_dir = "/content/runs/detect/predict/"
images = os.listdir(image_dir)[:10]

for img_file in images:
    img_path = os.path.join(image_dir, img_file)
    img = mpimg.imread(img_path)

    plt.imshow(img)
    plt.axis("off")
    plt.show()

## Model Export

Save trained model for reuse and deployment.

In [ ]:
!mkdir -p /content/my_Specialist_model

!cp /content/runs/detect/train/weights/best.pt /content/my_Specialist_model/my_Specialist_model.pt
!cp -r /content/runs/detect/train /content/my_Specialist_model/

%cd /content/my_Specialist_model
!zip /content/my_Specialist_model.zip my_Specialist_model.pt
!zip -r /content/my_Specialist_model.zip train
%cd /content

## Deployment Test

Load trained model and run prediction on a new image.

In [ ]:
import os
import zipfile
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from google.colab import files

print("Upload model zip file")
uploaded_zip = files.upload()
zip_path = list(uploaded_zip.keys())[0]

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/model")

model_path = "/content/model/my_Specialist_model.pt"

print("Upload test image")
uploaded_img = files.upload()
image_path = list(uploaded_img.keys())[0]

!yolo detect predict model="$model_path" source="$image_path" save=True

result_dir = "/content/runs/detect/predict"
result_images = [f for f in os.listdir(result_dir) if f.endswith((".jpg", ".png", ".jpeg"))]

result_path = os.path.join(result_dir, result_images[0])
img = mpimg.imread(result_path)

plt.imshow(img)
plt.axis("off")
plt.title("Predicted Resistor Value")
plt.show()

## Code Attribution

Portions of this notebook were adapted from external YOLO training workflows to support standard dataset preparation and model configuration tasks.

Specifically, the train/validation split script was adapted from the EdjeElectronics YOLO training workflow. The YAML configuration structure was also adapted from common Ultralytics YOLO dataset formatting practices.

These sections were used only for setup and workflow organization. The project-specific work, including dataset creation, resistor labeling logic, model purpose, training decisions, evaluation, and interpretation, was completed for this resistor recognition project.